<a href="https://colab.research.google.com/github/41371204h/1142-programming-language/blob/main/HW4_PTT_GoogleSheet_RAG%E6%95%B4%E7%90%86%E7%89%88.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# HW4：PTT → Google Sheet → RAG（整理版）

這份 notebook 保留完整流程：

1. 爬取 PTT movie 文章
2. 寫入指定 Google Sheet
3. 從 Google Sheet 讀回資料
4. 建立 FAISS RAG 索引
5. 用 Gemini 根據 PTT 資料回答問題

主要修正：原本設定了 `SHEET_URL`，但實際用 `gc.open(WORKSHEET_NAME)` 開啟試算表，容易打開錯的 Spreadsheet。新版固定使用 `gc.open_by_url(SHEET_URL)`。


In [14]:
!pip -q install gradio

In [1]:
# 安裝必要套件
!pip -q install gspread gspread-dataframe faiss-cpu sentence-transformers beautifulsoup4 requests google-generativeai


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 49.5 MB/s eta 0:00:00


In [2]:
import re
import time
import uuid
from datetime import datetime
from urllib.parse import urljoin

import requests
import pandas as pd
import numpy as np
from bs4 import BeautifulSoup

import faiss
from sentence_transformers import SentenceTransformer

import google.generativeai as genai
from google.colab import auth, userdata
import gspread
from google.auth import default
from gspread_dataframe import set_with_dataframe, get_as_dataframe


/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


## 1. 基本設定

請確認 `SHEET_URL` 是你要寫入的 Google Sheet。  
`PTT_WORKSHEET_NAME` 是存放 PTT 原始文章的分頁。


In [3]:
SHEET_URL = "https://docs.google.com/spreadsheets/d/1bMteJ88u-_o-6v1Ym1RZNDrdIpQA3_A6OucFpDQbyBs/edit?usp=sharing"
PTT_WORKSHEET_NAME = "ptt_movie_posts"
TIMEZONE_NOTE = "Asia/Taipei"

PTT_HEADER = [
    "post_id", "title", "url", "date", "author", "nrec",
    "created_at", "fetched_at", "content"
]

PTT_MOVIE_INDEX = "https://www.ptt.cc/bbs/movie/index.html"
PTT_COOKIES = {"over18": "1"}
USER_AGENT = "Mozilla/5.0 (compatible; Colab PTT crawler)"


## 2. 連線 Google Sheet

這裡是最重要的修正：使用 `open_by_url(SHEET_URL)`，不要用 worksheet 名稱打開 spreadsheet。


In [4]:
auth.authenticate_user()
creds, _ = default()
gc = gspread.authorize(creds)

# 關鍵修正：直接用網址開啟指定 Google Sheet
sh = gc.open_by_url(SHEET_URL)
print(f"✅ 已開啟試算表：{sh.title}")
print(f"🔗 {SHEET_URL}")


✅ 已開啟試算表：程式語言作業
🔗 https://docs.google.com/spreadsheets/d/1bMteJ88u-_o-6v1Ym1RZNDrdIpQA3_A6OucFpDQbyBs/edit?usp=sharing


In [5]:
def ensure_worksheet(spreadsheet, title, header, rows=1000):
    """取得或建立 worksheet，並確保表頭正確。"""
    try:
        ws = spreadsheet.worksheet(title)
    except gspread.WorksheetNotFound:
        ws = spreadsheet.add_worksheet(title=title, rows=str(rows), cols=str(len(header) + 5))
        ws.update([header])
        return ws

    values = ws.get_all_values()
    if not values:
        ws.update([header])
    elif values[0] != header:
        # 保留資料但重建欄位較危險，因此這裡直接清掉並重新建立正確表頭。
        # 若你要保留舊資料，請先備份 Google Sheet。
        ws.clear()
        ws.update([header])
    return ws


def read_sheet_df(ws, header):
    """從 worksheet 讀成 DataFrame，並清掉空列。"""
    df = get_as_dataframe(ws, evaluate_formulas=True, dtype=str).dropna(how="all")
    if df.empty:
        return pd.DataFrame(columns=header)
    df = df.loc[:, [c for c in df.columns if not str(c).startswith("Unnamed")]]
    for col in header:
        if col not in df.columns:
            df[col] = ""
    return df[header].fillna("")


def write_sheet_df(ws, df, header):
    """把 DataFrame 寫回 worksheet。"""
    df_out = df.copy()
    for col in header:
        if col not in df_out.columns:
            df_out[col] = ""
    df_out = df_out[header].infer_objects(copy=False).fillna("") # Add infer_objects to address FutureWarning

    # Google Sheet 寫入前統一轉字串，避免 Timestamp / NaN 型別問題
    for c in df_out.columns:
        df_out[c] = df_out[c].astype(str)

    ws.clear()
    set_with_dataframe(ws, df_out, include_index=False, include_column_header=True, resize=True)
    return len(df_out)


ws_ptt = ensure_worksheet(sh, PTT_WORKSHEET_NAME, PTT_HEADER)
print(f"✅ 已準備 worksheet：{ws_ptt.title}")

✅ 已準備 worksheet：ptt_movie_posts


## 3. PTT movie 爬蟲

這段只負責爬 PTT，不碰 RAG。資料會先存在 `new_posts_df`。


In [6]:
def now_iso():
    return datetime.now().isoformat(timespec="seconds")


def get_soup(url):
    resp = requests.get(
        url,
        timeout=20,
        headers={"User-Agent": USER_AGENT},
        cookies=PTT_COOKIES,
    )
    resp.raise_for_status()
    return BeautifulSoup(resp.text, "html.parser")


def get_prev_index_url(soup):
    for a in soup.select("div.btn-group-paging a.btn.wide"):
        if "上頁" in a.get_text(strip=True):
            href = a.get("href")
            return urljoin("https://www.ptt.cc", href) if href else None
    return None


def parse_nrec(nrec_span):
    if not nrec_span:
        return 0
    txt = nrec_span.get_text(strip=True)
    if txt == "爆":
        return 100
    if txt.startswith("X"):
        try:
            return -int(txt[1:])
        except Exception:
            return -10
    try:
        return int(txt)
    except Exception:
        return 0


def extract_post_list(index_soup):
    posts = []
    for item in index_soup.select("div.r-ent"):
        a = item.select_one("div.title a")
        if not a:
            continue

        title = a.get_text(strip=True)
        url = urljoin("https://www.ptt.cc", a.get("href"))
        author_node = item.select_one("div.author")
        date_node = item.select_one("div.date")
        nrec_node = item.select_one("div.nrec span")

        posts.append({
            "title": title,
            "url": url,
            "author": author_node.get_text(strip=True) if author_node else "",
            "date": date_node.get_text(strip=True) if date_node else "",
            "nrec": parse_nrec(nrec_node),
        })
    return posts


def clean_ptt_content(article_soup):
    main = article_soup.select_one("#main-content")
    if not main:
        return "", ""

    # 取出文章建立時間
    created_at = ""
    metalines = main.select("div.article-metaline")
    for m in metalines:
        tag = m.select_one("span.article-meta-tag")
        value = m.select_one("span.article-meta-value")
        if tag and value and tag.get_text(strip=True) == "時間":
            created_at = value.get_text(strip=True)

    # 移除 meta 與推文
    for node in main.select("div.article-metaline, div.article-metaline-right, div.push"):
        node.decompose()

    text = main.get_text("\n", strip=True)
    text = re.split(r"\n--\n|\n※ 發信站:", text)[0].strip()
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text, created_at


def make_post_id(url):
    # 用文章網址檔名當 post_id，穩定且方便去重
    return url.rstrip("/").split("/")[-1].replace(".html", "")


def crawl_ptt_movie(pages=2, delay=0.5):
    """爬取 PTT movie 最新 pages 頁文章。"""
    all_rows = []
    index_url = PTT_MOVIE_INDEX

    for page in range(int(pages)):
        print(f"📄 正在讀取列表頁 {page + 1}/{pages}: {index_url}")
        index_soup = get_soup(index_url)
        post_list = extract_post_list(index_soup)

        for p in post_list:
            try:
                article_soup = get_soup(p["url"])
                content, created_at = clean_ptt_content(article_soup)
                row = {
                    "post_id": make_post_id(p["url"]),
                    "title": p["title"],
                    "url": p["url"],
                    "date": p["date"],
                    "author": p["author"],
                    "nrec": p["nrec"],
                    "created_at": created_at,
                    "fetched_at": now_iso(),
                    "content": content,
                }
                all_rows.append(row)
                time.sleep(delay)
            except Exception as e:
                print(f"⚠️ 跳過文章：{p.get('title', '')}，原因：{e}")

        prev_url = get_prev_index_url(index_soup)
        if not prev_url:
            break
        index_url = prev_url
        time.sleep(delay)

    df = pd.DataFrame(all_rows, columns=PTT_HEADER)
    print(f"✅ 本次爬到 {len(df)} 篇文章")
    return df

## 4. 執行爬蟲並寫入 Google Sheet

這一格會：

1. 從 Google Sheet 讀取既有資料
2. 爬取新的 PTT 資料
3. 合併並用 `post_id` 去重
4. 寫回 Google Sheet
5. 再讀一次確認真的寫入成功


In [8]:
# 你可以調整 pages，例如 pages=1 先測試，確認成功後再改成 3 或 5
new_posts_df = crawl_ptt_movie(pages=2, delay=1.0)

old_posts_df = read_sheet_df(ws_ptt, PTT_HEADER)
print(f"📌 Google Sheet 原本有 {len(old_posts_df)} 筆")

ptt_posts_df = pd.concat([old_posts_df, new_posts_df], ignore_index=True)
ptt_posts_df = ptt_posts_df.drop_duplicates(subset=["post_id"], keep="last")
ptt_posts_df = ptt_posts_df.sort_values(by="fetched_at", ascending=False)

written_count = write_sheet_df(ws_ptt, ptt_posts_df, PTT_HEADER)
print(f"✅ 已寫入 Google Sheet：{written_count} 筆")

verify_df = read_sheet_df(ws_ptt, PTT_HEADER)
print(f"🔍 從 Google Sheet 重新讀回：{len(verify_df)} 筆")

if len(verify_df) == written_count:
    print("✅ 寫入驗證成功")
else:
    print("⚠️ 寫入筆數與讀回筆數不同，請檢查 Google Sheet 權限或資料格式")

📄 正在讀取列表頁 1/2: https://www.ptt.cc/bbs/movie/index.html
📄 正在讀取列表頁 2/2: https://www.ptt.cc/bbs/movie/index11001.html
✅ 本次爬到 26 篇文章
📌 Google Sheet 原本有 59 筆
✅ 已寫入 Google Sheet：59 筆
🔍 從 Google Sheet 重新讀回：59 筆
✅ 寫入驗證成功


## 5. 從 Google Sheet 建立 RAG 索引

重點：RAG 不直接吃剛爬下來的記憶體資料，而是**從 Google Sheet 重新讀回**，這樣才能確認流程真的是：

`PTT → Google Sheet → RAG`


In [9]:
# 從 Google Sheet 重新讀取，作為 RAG 的唯一資料來源
rag_source_df = read_sheet_df(ws_ptt, PTT_HEADER)

# 清掉沒有內容的文章
rag_source_df = rag_source_df[rag_source_df["content"].astype(str).str.strip() != ""].copy()
print(f"📚 可用於 RAG 的文章數：{len(rag_source_df)}")

rag_source_df.head()


📚 可用於 RAG 的文章數：59


,post_id,title,url,date,author,nrec,created_at,fetched_at,content
0,M.1780934290.A.449,[新聞] 愛上同學錯了嗎？《偶像禁愛令》揭日娛殘酷黑幕,https://www.ptt.cc/bbs/movie/M.1780934290.A.44...,6/8,hihihihehehe,1,Mon Jun 8 23:58:07 2026,2026-06-09 10:30:31,愛上同學錯了嗎？《偶像禁愛令》揭日娛殘酷黑幕\n\n2026/06/08 21:47\n\n...
1,M.1780932908.A.918,[新聞]超級瑪利歐銀河成本1.1億美元！票房逾10億,https://www.ptt.cc/bbs/movie/M.1780932908.A.91...,6/8,XDGEE,9,Mon Jun 8 23:35:06 2026,2026-06-09 10:30:29,《超級瑪利歐銀河》成本1.1億美元！票房逾10億！\n截至當地時間週日（7日），《超級瑪利歐...
2,M.1780932869.A.A74,[新聞]電影院提前曝光《蜘蛛人：重生日》片長！,https://www.ptt.cc/bbs/movie/M.1780932869.A.A7...,6/8,XDGEE,14,Mon Jun 8 23:34:26 2026,2026-06-09 10:30:28,電影院提前曝光《蜘蛛人：重生日》片長！打破 MCU 蜘蛛人電影紀錄？！\n\n《蜘蛛人：重生...
3,M.1780932359.A.BD0,[好雷] 失樂園,https://www.ptt.cc/bbs/movie/M.1780932359.A.BD...,6/8,smallroad,4,Mon Jun 8 23:25:50 2026,2026-06-09 10:30:26,雷文防雷資訊頁\n~*-*~*-*~*-*~*-*~*-*~*-*~*-*~*-*~*-*~...
4,M.1780930618.A.6CC,[新聞] MCU蜘蛛人：重生日第二支預告外流遭索尼,https://www.ptt.cc/bbs/movie/M.1780930618.A.6C...,6/8,pl132,11,Mon Jun 8 22:56:54 2026,2026-06-09 10:30:25,MCU《蜘蛛人：重生日》第二支預告外流遭索尼強制下架 浩克劇情疑與「玩具暴雷」互\n相印證\...


In [10]:
print("正在載入多語言 Embedding 模型...")
embedding_model = SentenceTransformer("BAAI/bge-m3")
print("✅ Embedding 模型載入完成")


def build_rag_documents(df):
    docs = []
    for _, row in df.iterrows():
        title = str(row.get("title", ""))
        content = str(row.get("content", ""))
        url = str(row.get("url", ""))
        author = str(row.get("author", ""))
        date = str(row.get("date", ""))
        nrec = str(row.get("nrec", ""))

        text = (f"標題：{title}\n"
                f"作者：{author}\n"
                f"日期：{date}\n"
                f"推文數：{nrec}\n"
                f"內容：{content}")
        docs.append({
            "post_id": str(row.get("post_id", "")),
            "title": title,
            "url": url,
            "text": text,
        })
    return docs


def build_faiss_index(docs):
    if not docs:
        raise ValueError("沒有可建立索引的文件")

    texts = [d["text"] for d in docs]

    print("正在使用 BAAI/bge-m3 進行分批向量編碼...")
    # 優化關鍵 1：加入 batch_size=4 或 8，防止記憶體一次吃太滿
    # 優化關鍵 2：bge-m3 支援彈性維度，如果真的太慢，可以用 max_length 限制文字長度
    embeddings = embedding_model.encode(
        texts,
        batch_size=8,          # 分批處理，每次只算 8 篇
        convert_to_numpy=True,
        show_progress_bar=True
    )
    embeddings = embeddings.astype("float32")

    # 正規化（維持 L2 距離等同於餘弦相似度）
    faiss.normalize_L2(embeddings)

    index = faiss.IndexFlatL2(embeddings.shape[1])
    index.add(embeddings)
    return index, embeddings


rag_documents = build_rag_documents(rag_source_df)
rag_index, rag_embeddings = build_faiss_index(rag_documents)

print(f"✅ RAG 索引建立完成：{len(rag_documents)} 篇文章，向量維度 {rag_embeddings.shape[1]}")

正在載入多語言 Embedding 模型...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/15.8k [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

✅ Embedding 模型載入完成
正在使用 BAAI/bge-m3 進行分批向量編碼...


Batches:   0%|          | 0/8 [00:00<?, ?it/s]

✅ RAG 索引建立完成：59 篇文章，向量維度 1024


## 6. Gemini 設定與 RAG 問答

請先在 Colab Secrets 裡建立 `gemini`，內容是你的 Gemini API key。


In [11]:
api_key = userdata.get("gemini")
if not api_key:
    raise ValueError("找不到 Colab Secret：gemini。請先在 Colab Secrets 新增 Gemini API key。")

genai.configure(api_key=api_key)

# 若你的帳號不支援這個模型，可改成你可用的 Gemini model name
GEMINI_MODEL_NAME = "gemini-3-flash-preview"
llm = genai.GenerativeModel(GEMINI_MODEL_NAME)
print(f"✅ Gemini 已設定：{GEMINI_MODEL_NAME}")


✅ Gemini 已設定：gemini-3-flash-preview


In [12]:
def retrieve_docs(query, k=3):
    if rag_index is None or not rag_documents:
        return []
    q_emb = embedding_model.encode([query], convert_to_numpy=True).astype("float32")

    # 【優化】問題向量也要正規化
    faiss.normalize_L2(q_emb)

    distances, indices = rag_index.search(q_emb, int(k))

    results = []
    for dist, idx in zip(distances[0], indices[0]):
        if idx == -1:
            continue
        doc = rag_documents[idx].copy()
        doc["distance"] = float(dist)
        results.append(doc)
    return results


def query_rag(question, k=3):
    docs = retrieve_docs(question, k=k)
    if not docs:
        return "找不到相關 PTT 資料。"

    context = "\n\n---\n\n".join(
        [f"來源標題：{d['title']}\n來源網址：{d['url']}\n{d['text']}" for d in docs]
    )

    prompt = f"""
你是一個根據 PTT 電影版資料回答問題的助教。
請只根據【PTT 資料】回答。
如果資料不足，請明確說「目前資料不足，無法判斷」，不要自行編造。
回答請使用繁體中文，並在最後列出參考來源標題與網址。

【PTT 資料】
{context}

【問題】
{question}

【回答】
""".strip()

    response = llm.generate_content(prompt)
    return response.text

## 7. 快速測試


In [13]:
question = input("請輸入問題：")
answer = query_rag(question, k=3)
print(answer)


請輸入問題：電影推薦
根據您提供的【PTT 資料】，整理出的電影推薦如下：

### 1. 血腥暴力與劇情兼具（類似《冰冷熱帶魚》類型）
如果您尋找的是含有血腥、暴力元素且具備劇情的電影，資料中提到的相關作品包括：
*   **《冰冷熱帶魚》**
*   **《人肉叉燒包》**（港片）

### 2. 6/7 當週上映新片推薦（依類型區分）
根據 6/7 當週新片預告排行榜與分類，有以下不同類型的選擇：
*   **驚悚懸疑：**《夜勤事件》、《殘殺路人甲》
*   **犯罪成長：**《盜音天才》
*   **動作冒險：**《太空超人》
*   **惡搞諷刺：**《驚聲尖笑 6》
*   **愛情故事：**《妳最後留下的歌》
*   **人性戰爭：**《紐倫堡》
*   **溫馨勵志：**《祈憐之歌》、《寶貝對不起》
*   **動畫漫畫：**《神奇數字馬戲團：最終章》
*   **演唱會電影：**《PLAVE Asia Tour》

### 3. 紀錄片推薦
*   **《飛吧！熊鷹》**：這是一部關於台灣猛禽熊鷹的紀錄片，內容涵蓋生態知識、原民文化與保育衝突，兼具深度與幽默感，獲得網友給予「好」的無雷評價。

---
**參考來源：**
1. [片單] 類似冰冷熱帶魚 (https://www.ptt.cc/bbs/movie/M.1779066370.A.09E.html)
2. [情報] 6/7 當週11部新片預告+Youtube觀看排行 (https://www.ptt.cc/bbs/movie/M.1780848439.A.ADC.html)
3. [ 好無雷] 飛吧！熊鷹 (https://www.ptt.cc/bbs/movie/M.1779121114.A.496.html)


## 常見錯誤檢查

如果 PTT 資料沒有寫回 Google Sheet，請依序檢查：

1. 是否有成功印出 `已開啟試算表`，且名稱正確。
2. `SHEET_URL` 是否是你要寫入的那一份 Google Sheet。
3. Google Sheet 權限是否允許目前 Colab 登入的 Google 帳號編輯。
4. 是否執行到「執行爬蟲並寫入 Google Sheet」那一格。
5. 是否有看到 `寫入驗證成功`。
6. RAG 要從 `rag_source_df = read_sheet_df(...)` 開始，確保資料來源是 Google Sheet，而不是記憶體中的暫存變數。


In [19]:
import gradio as gr

def query_rag_with_analytics(question, k=3):
    # 1. 檢索相關文章
    docs = retrieve_docs(question, k=int(k))
    if not docs:
        return "找不到相關 PTT 資料。", "N/A", "N/A"

    # 2. 推文熱度分析 (利用資料庫中的 nrec)
    nrec_list = []
    for d in docs:
        # 從 text 中或原始資料尋找推文數（這裡從 text 裡用正規表達式撈出數字，或預防型處理）
        match = re.search(r"推文數：(-?\d+)", d["text"])
        if match:
            nrec_list.append(int(match.group(1)))
        else:
            nrec_list.append(0)

    avg_nrec = sum(nrec_list) / len(nrec_list)

    # 根據平均推文數給予熱度評價
    if avg_nrec >= 80:
        popularity_status = f"🔥 爆熱門 (平均推文：{avg_nrec:.1f})"
    elif avg_nrec >= 30:
        popularity_status = f"📈 討論度高 (平均推文：{avg_nrec:.1f})"
    elif avg_nrec >= 10:
        popularity_status = f"💬 有些許討論 (平均推文：{avg_nrec:.1f})"
    else:
        popularity_status = f"❄️ 冷清/個案討論 (平均推文：{avg_nrec:.1f})"

    # 3. 建立 Context
    context = "\n\n---\n\n".join(
        [f"來源標題：{d['title']}\n來源網址：{d['url']}\n{d['text']}" for d in docs]
    )

    # 4. 設計 Prompt（強迫 Gemini 同時輸出答案與情緒分析）
    prompt = f"""
你是一個根據 PTT 電影版資料回答問題並分析輿情的助教。
請執行以下兩項任務：
1. 根據【PTT 資料】回答【問題】。如果資料不足，請明確說「目前資料不足，無法判斷」，不要自行編造。
2. 分析【PTT 資料】中網民的「整體情緒指標」（例如：普遍支持、強烈負雷、酸民諷刺、理性討論等），並給出一個主要的情緒標籤。

回答請使用繁體中文，並在最後列出參考來源標題與網址。

【PTT 資料】
{context}

【問題】
{question}

【回答與情緒指標】
""".strip()

    # 5. 呼叫 Gemini
    response = llm.generate_content(prompt)

    # 6. 利用第二個 Prompt 讓 Gemini 獨立吐出精準的「情緒指標標籤」，方便放在 Gradio 的獨立欄位中
    sentiment_prompt = f"請根據以下 PTT 內容，只輸出一個最適合的四字情緒標籤（例如：正評如潮、虛張聲勢、酸氣沖天、兩極分化、平淡無奇）：\n\n{context}"
    sentiment_tag = llm.generate_content(sentiment_prompt).text.strip()

    return response.text, sentiment_tag, popularity_status


# ================= 7. Gradio 介面設計 =================

with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 🎬 PTT 電影版 RAG 智能輿情分析系統")
    gr.Markdown("輸入你想查詢的電影或話題，系統會從 Google Sheet 資料庫檢索 PTT 文章，並結合 Gemini 進行回答、情緒與熱度分析！")

    with gr.Row():
        with gr.Column(scale=2):
            input_query = gr.Textbox(label="請輸入你的問題", placeholder="例如：網友對《沙丘2》的評價如何？", lines=2)
            slider_k = gr.Slider(minimum=1, maximum=5, value=3, step=1, label="檢索文章篇數 (K 值)")
            btn_submit = gr.Button("開始檢索與分析", variant="primary")

        with gr.Column(scale=1):
            # 獨立的分析數據儀表板
            output_sentiment = gr.Textbox(label="📊 網民情緒指標", interactive=False)
            output_popularity = gr.Textbox(label="🔥 PTT 討論熱度", interactive=False)

    with gr.Row():
        output_answer = gr.Markdown(label="🤖 RAG 回答結果")

    # 綁定按鈕事件
    btn_submit.click(
        fn=query_rag_with_analytics,
        inputs=[input_query, slider_k],
        outputs=[output_answer, output_sentiment, output_popularity]
    )

# 啟動 Gradio (在 Colab 中會自動產生內嵌網頁與外部連結)
demo.launch(debug=True)

/tmp/ipykernel_2399/1495561544.py:66: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as demo:


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://f8ff25f184a2aba1bc.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://f8ff25f184a2aba1bc.gradio.live
